# Notebook 4: Inner Join Kaiaulu Comments with Sentiment Labels

By now, you should have:
1. The contextualized Gold Standard dataset from Notebook 2 (`github_gold_standard_contextualized.csv`)
2. Comment data freshly downloaded from GitHub via Kaiaulu (`{repo}_commit_comments.csv` and `{repo}_pr_inline_comments.csv`)

Neither is complete on its own. The Gold Standard has sentiment labels, but no fresh GitHub metadata. Kaiaulu's output has GitHub metadata, but no sentiment labels. INNER JOINing them on `comment_id` gives us both.

### Before you start

Three files need to be in `data/` before running any cells:

| File | Where it comes from |
|---|---|
| `data/github_gold_standard_contextualized.csv` | Output of Notebook 2, Step 7 |
| `data/{repo}_commit_comments.csv` | Copy from `rawdata/github/{owner}/{repo}/commit_comments/` after running `vignettes/download_github_events.Rmd` |
| `data/{repo}_pr_inline_comments.csv` | Copy from `rawdata/github/{owner}/{repo}/pr_comments/` after running `vignettes/download_github_pull_request_comments.Rmd` |

If either Kaiaulu CSV is missing, go back and run the corresponding vignette in Notebook 3 first.

### Step 1: Import dependencies

In [1]:
import os
import pandas as pd

### Step 2: Configure Project

Set `OWNER` and `REPO` to match the project you ran the Kaiaulu vignettes for. These must match the `owner` and `repo` columns in `github_gold_standard_contextualized.csv` and the filenames of the Kaiaulu output CSVs in `data/`.

In [ ]:
# Configure these before running
OWNER = "ADD_OWNER_HERE"   # GitHub repo owner (must match 'owner' column in contextualized CSV)
REPO  = "ADD_REPO_HERE"   # GitHub repo name  (must match 'repo'  column in contextualized CSV)

DATA_DIR = os.path.join(os.path.dirname(os.getcwd()), "data")

### Step 3: Load the contextualized dataset

Load the full contextualized Gold Standard dataset from Notebook 2.

In [ ]:
contextualized = pd.read_csv(os.path.join(DATA_DIR, "github_gold_standard_contextualized.csv"))
print(f"Full contextualized dataset: {len(contextualized)} rows")

project_ctx = contextualized[(contextualized['owner'] == OWNER) & (contextualized['repo'] == REPO)].copy()
print(f"{OWNER}/{REPO} rows: {len(project_ctx)}")

### Step 4: Load the Kaiaulu output CSVs

Load the two CSVs you copied into `data/` after running the Kaiaulu vignettes.

In [9]:
kaiaulu_commit = pd.read_csv(os.path.join(DATA_DIR, f"{REPO}_commit_comments.csv"))
kaiaulu_pr     = pd.read_csv(os.path.join(DATA_DIR, f"{REPO}_pr_inline_comments.csv"))

print(f"Kaiaulu commit comments   : {len(kaiaulu_commit)} rows, columns: {list(kaiaulu_commit.columns)}")
print(f"Kaiaulu PR inline comments: {len(kaiaulu_pr)} rows, columns: {list(kaiaulu_pr.columns)}")

Kaiaulu commit comments   : 1569 rows, columns: ['comment_id', 'commit_id', 'author_login', 'author_id', 'body', 'created_at', 'updated_at']
Kaiaulu PR inline comments: 6100 rows, columns: ['review_id', 'comment_id', 'html_url', 'created_at', 'updated_at', 'comment_user_login', 'author_association', 'file_path', 'start_line', 'line', 'original_start_line', 'original_line', 'position', 'diff_hunk', 'body', 'commit_id']


### Step 5: INNER JOIN - Commit Comments

Join the project's contextualized Gold Standard rows against Kaiaulu's commit comments on `comment_id`.

In [ ]:
commit_joined = project_ctx.merge(
    kaiaulu_commit,
    on='comment_id',
    how='inner', # INNER JOIN
    suffixes=('_gold', '_kaiaulu')
)

commit_dropped = len(project_ctx) - len(commit_joined)
print(f"{OWNER}/{REPO} rows in contextualized dataset : {len(project_ctx)}")
print(f"Rows matched in Kaiaulu commit comments       : {len(commit_joined)}")
print(f"Rows not found in Kaiaulu download            : {commit_dropped}")
if commit_dropped > 0:
    print(f"  These {commit_dropped} comment IDs exist in the Gold Standard but were not found in the Kaiaulu download.")

print("\nJoined commit comments (first 5 rows):")
display(commit_joined.head())

out_path = os.path.join(DATA_DIR, f"{REPO}_sentiment_commit_comments_joined.csv")
commit_joined.to_csv(out_path, index=False)
print(f"\nSaved: {out_path}")

### Step 6: INNER JOIN - PR inline comments

Same join as step 5, but against Kaiaulu's PR inline comments.

In [ ]:
pr_joined = project_ctx.merge(
    kaiaulu_pr,
    on='comment_id',
    how='inner', # INNER JOIN
    suffixes=('_gold', '_kaiaulu')
)

pr_dropped = len(project_ctx) - len(pr_joined)
print(f"{OWNER}/{REPO} rows in contextualized dataset : {len(project_ctx)}")
print(f"Rows matched in Kaiaulu PR inline comments    : {len(pr_joined)}")
print(f"Rows not found in Kaiaulu download            : {pr_dropped}")
if pr_dropped > 0:
    print(f"  These {pr_dropped} comment IDs exist in the Gold Standard but were not found in the Kaiaulu download.")

print("\nJoined PR inline comments (first 5 rows):")
display(pr_joined.head())

out_path = os.path.join(DATA_DIR, f"{REPO}_sentiment_pr_inline_comments_joined.csv")
pr_joined.to_csv(out_path, index=False)
print(f"\nSaved: {out_path}")

### You're done!

**To run for a different project:** update `OWNER` and `REPO` in Step 2, copy the new Kaiaulu CSVs into `data/`, and re-run Steps 3–6.